# ECCT on LDPC(49,24)

This notebook clones the repository, checks the Kaggle GPU, trains ECCT on the same LDPC code and model dimensions as AECCT, and prints the resulting BER/FER log.

Enable **Internet** and a **GPU accelerator** in Kaggle before running. `ECCT_EPOCHS=2000` matches AECCT's 1,000 normal-training plus 1,000 quantization-aware-training epochs. Set it to 1000 if you want the direct phase-1 comparison instead.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/gouravanirudh05/SRIP_LDPC_Decoding_using_Machine_Learning.git'
REPO_DIR = Path('/kaggle/working/ldpc_repo')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

ECCT_DIR = REPO_DIR / 'ECCT'
if not (ECCT_DIR / 'Main.py').exists():
    
    raise FileNotFoundError('ECCT/Main.py is missing from the cloned repository.')

os.chdir(ECCT_DIR)
print('Working directory:', Path.cwd())

Cloning into '/kaggle/working/ldpc_repo'...


Working directory: /kaggle/working/ldpc_repo/ECCT


In [2]:
!pwd
!ls -lah
!ls -lah /kaggle/working/ldpc_repo
!find . -maxdepth 2 -type f | sort

/kaggle/working/ldpc_repo/ECCT
total 48K
drwxr-xr-x 3 root root 4.0K Jul 27 03:52 .
drwxr-xr-x 7 root root 4.0K Jul 27 03:52 ..
drwxr-xr-x 2 root root 4.0K Jul 27 03:52 Codes_DB
-rw-r--r-- 1 root root 4.5K Jul 27 03:52 Codes.py
-rw-r--r-- 1 root root 1.1K Jul 27 03:52 LICENSE
-rw-r--r-- 1 root root  11K Jul 27 03:52 Main.py
-rw-r--r-- 1 root root 5.9K Jul 27 03:52 Model.py
-rw-r--r-- 1 root root 2.0K Jul 27 03:52 README.md
total 40K
drwxr-xr-x 7 root root 4.0K Jul 27 03:52 .
drwxr-xr-x 4 root root 4.0K Jul 27 03:52 ..
drwxr-xr-x 3 root root 4.0K Jul 27 03:52 5G-Decoder-main
drwxr-xr-x 4 root root 4.0K Jul 27 03:52 AECCT-main
drwxr-xr-x 3 root root 4.0K Jul 27 03:52 BasicImplementation-Week1
drwxr-xr-x 3 root root 4.0K Jul 27 03:52 ECCT
drwxr-xr-x 8 root root 4.0K Jul 27 03:52 .git
-rw-r--r-- 1 root root 4.6K Jul 27 03:52 .gitignore
-rw-r--r-- 1 root root   38 Jul 27 03:52 README.md
./Codes_DB/BCH_N31_K16.txt
./Codes_DB/BCH_N63_K36.txt
./Codes_DB/BCH_N63_K45.txt
./Codes_DB/BCH_N63_K51.t

In [3]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'tqdm'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. In Kaggle, select GPU under Notebook options.')
print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Training configuration

The code, rate, optimizer, batch size, seed, number of blocks, and embedding dimension match the AECCT run. ECCT is run for 2,000 epochs to match AECCT's total number of training epochs across both phases.

In [ ]:
import time

ECCT_EPOCHS = 1000  # Change to 1000 for comparison with AECCT phase 1 only.
command = [
    sys.executable, 'Main.py',
    '--gpus=0',
    f'--epochs={ECCT_EPOCHS}',
    '--workers=4',
    '--lr=1e-4',
    '--batch_size=128',
    '--test_batch_size=512',
    '--seed=42',
    '--code_type=LDPC',
    '--code_n=49',
    '--code_k=24',
    '--N_dec=6',
    '--d_model=128',
    '--h=8',
]
print('Running:', ' '.join(command))
start = time.perf_counter()
subprocess.run(command, cwd=str(ECCT_DIR), check=True)
print(f'Total wall time: {(time.perf_counter() - start) / 3600:.2f} hours')

Running: /usr/bin/python3 Main.py --gpus=1 --epochs=1000 --workers=4 --lr=1e-4 --batch_size=128 --test_batch_size=512 --seed=42 --code_type=LDPC --code_n=49 --code_k=24 --N_dec=6 --d_model=128 --h=8


Path to model/logs: Results_ECCT/LDPC__Code_n_49_k_24__27_07_2026_03_52_55
Namespace(epochs=1000, workers=4, lr=0.0001, gpus='1', batch_size=128, test_batch_size=512, seed=42, code_type='LDPC', code_k=24, code_n=49, standardize=False, N_dec=6, d_model=128, h=8, code=<__main__.Code object at 0x7e5b2f1381a0>, path='Results_ECCT/LDPC__Code_n_49_k_24__27_07_2026_03_52_55')
Self-Attention Sparsity Ratio=72.26%, Self-Attention Complexity Ratio=13.87%
Mask:
 tensor([[[[False,  True,  True,  ...,  True,  True,  True],
          [ True, False,  True,  ...,  True,  True,  True],
          [ True,  True, False,  ...,  True,  True,  True],
          ...,
          [ True,  True,  True,  ..., False,  True,  True],
          [ True,  True,  True,  ...,  True, False,  True],
          [ True,  True,  True,  ...,  True,  True, False]]]])
ECC_Transformer(
  (decoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attn): MultiHeadedAttention(
          (linears): Module

FER count threshold reached for EbN0:4
Test EbN0=4, BER=4.23e-02
FER count threshold reached for EbN0:5
Test EbN0=5, BER=2.01e-02



Test Loss 4: 1.24e-01 5: 6.03e-02 6: 2.36e-02
Test FER 4: 6.14e-01 5: 3.71e-01 6: 1.69e-01
Test BER 4: 4.23e-02 5: 2.01e-02 6: 7.41e-03
Test -ln(BER) 4: 3.16e+00 5: 3.91e+00 6: 4.91e+00
# of testing samples: [100352.0, 100352.0, 100352.0]
 Test Time 153.11240649223328 s



FER count threshold reached for EbN0:6
Test EbN0=6, BER=7.41e-03


Training epoch 2, Batch 500/1000: LR=1.00e-04, Loss=1.25e-01 BER=4.23e-02 FER=5.23e-01
Training epoch 2, Batch 1000/1000: LR=1.00e-04, Loss=1.21e-01 BER=4.13e-02 FER=5.10e-01
Epoch 2 Train Time 68.9194610118866s

Training epoch 3, Batch 500/1000: LR=1.00e-04, Loss=1.12e-01 BER=3.89e-02 FER=4.79e-01
Training epoch 3, Batch 1000/1000: LR=1.00e-04, Loss=1.09e-01 BER=3.82e-02 FER=4.73e-01
Epoch 3 Train Time 67.86445140838623s

Training epoch 4, Batch 500/1000: LR=1.00e-04, Loss=1.02e-01 BER=3.62e-02 FER=4.59e-01
Training epoch 4, Batch 1000/1000: LR=1.00e-04, Loss=9.97e-02 BER=3.56e-02 FER=4.53e-01
Epoch 4 Train Time 67.46910262107849s

Training epoch 5, Batch 500/1000: LR=1.00e-04, Loss=9.31e-02 BER=3.39e-02 FER=4.36e-01
Training epoch 5, Batch 1000/1000: LR=1.00e-04, Loss=9.15e-02 BER=3.35e-02 FER=4.32e-01
Epoch 5 Train Time 67.53309512138367s

Training epoch 6, Batch 500/1000: LR=1.00e-04, Loss=8.54e-02 BER=3.16e-02 FER=4.13e-01
Training epoch 6, Batch 1000/1000: LR=1.00e-04, Loss=8.46e

In [ ]:
# Print the latest ECCT result directory and its final log lines.
result_dirs = sorted((ECCT_DIR / 'Results_ECCT').glob('*'), key=lambda p: p.stat().st_mtime)
if not result_dirs:
    raise FileNotFoundError('No ECCT result directory was produced.')
latest = result_dirs[-1]
log_file = latest / 'logging.txt'
print('Result directory:', latest)
print('Checkpoint:', latest / 'best_model')
print('\n'.join(log_file.read_text(errors='replace').splitlines()[-40:]))